In [1]:
import time
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

from aeon.classification.interval_based import RSTSF
from tscglue.interval_models import RSTSFRandom, RSTSFUnsupervised, RSTSFUnsupervisedRaw
from tscglue import data_loader


In [2]:
from tscglue import data_loader
from tscglue.interval_models import RSTSFUnsupervised, RSTSFCombined, RSTSFUnsupervisedPerRep

# Use a larger dataset to see realistic shapes
dataset = 'Fish'
dataset = 'InsectWingbeatSound'
X_train, y_train, X_test, y_test = data_loader.load_fold(dataset, fold=5)
print(f"X_train: {X_train.shape}")

print("\n--- RSTSFUnsupervised fit (prints shapes before/after GRP) ---")
m = RSTSFUnsupervised(n_estimators=2, n_intervals=50, random_state=42, n_jobs=-1, verbose=True)
m.fit(X_train, y_train)

print("\n--- RSTSFCombined fit (prints shapes before/after GRP) ---")
m2 = RSTSFCombined(n_estimators=2, n_intervals_random=600, n_intervals_unsupervised=50, random_state=42, n_jobs=-1, verbose=True)
m2.fit(X_train, y_train)

print("\n--- RSTSFUnsupervisedPerRep fit (per-rep GRP, batched) ---")
m3 = RSTSFUnsupervisedPerRep(n_estimators=2, n_intervals=50, batch_size=100, random_state=42, n_jobs=-1, verbose=True)
m3.fit(X_train, y_train)

X_train: (220, 1, 256)

--- RSTSFUnsupervised fit (prints shapes before/after GRP) ---
[RSTSFUnsupervised] rep 0 features: (220, 60256) (101.1 MB)
[RSTSFUnsupervised] rep 1 features: (220, 60008) (100.7 MB)
[RSTSFUnsupervised] rep 2 features: (220, 29322) (49.2 MB)
[RSTSFUnsupervised] rep 3 features: (220, 1846) (3.1 MB)
[RSTSFUnsupervised] Xt before GRP: (220, 151432) (254.2 MB)
[RSTSFUnsupervised] Xt after GRP: (220, 4623) (7.8 MB)

--- RSTSFCombined fit (prints shapes before/after GRP) ---
[RSTSFCombined] rep 0 rand features: (220, 4123) (6.9 MB)
[RSTSFCombined] rep 0 unsup features: (220, 60256) (101.1 MB)
[RSTSFCombined] rep 1 rand features: (220, 4130) (6.9 MB)
[RSTSFCombined] rep 1 unsup features: (220, 60008) (100.7 MB)
[RSTSFCombined] rep 2 rand features: (220, 3969) (6.7 MB)
[RSTSFCombined] rep 2 unsup features: (220, 29322) (49.2 MB)
[RSTSFCombined] rep 3 rand features: (220, 609) (1.0 MB)
[RSTSFCombined] rep 3 unsup features: (220, 1846) (3.1 MB)
[RSTSFCombined] Xt_rand bef

,n_estimators,2
,n_intervals,50
,min_interval_length,3
,n_components,'auto'
,batch_size,100
,estimator,None
,random_state,42
,n_jobs,-1
,verbose,True


In [3]:
# Smaller UCR datasets
DATASETS = [
    "ArrowHead",      # 36 train,  175 test, len=251, 3 classes
    "Coffee",         # 28 train,   28 test, len=286, 2 classes
    "BME",            # 30 train,  150 test, len=128, 3 classes
    "ECG200",         # 100 train, 100 test, len=96,  2 classes
    "CBF",            # 30 train,  900 test, len=128, 3 classes
    "Beef",           # 30 train,   30 test, len=470, 5 classes
    "OliveOil",       # 30 train,   30 test, len=570, 4 classes
    "Wine",           # 57 train,   54 test, len=234, 2 classes
    "Plane",          # 105 train, 105 test, len=144, 7 classes
    "GunPoint",       # 50 train,  150 test, len=150, 2 classes
    "FaceFour",       # 24 train,   88 test, len=350, 4 classes
    "Fish",           # 175 train, 175 test, len=463, 7 classes
    "Trace",          # 100 train, 100 test, len=275, 4 classes
    "SyntheticControl", # 300 train, 300 test, len=60, 6 classes
    "ItalyPowerDemand", # 67 train,  1029 test, len=24, 2 classes
]

CLASSIFIERS = [
    ("RSTSF",                  lambda: RSTSF(n_estimators=200, n_intervals=50, random_state=42, n_jobs=-1)),
    ("RSTSF-Random",           lambda: RSTSFRandom(n_estimators=200, n_intervals=600, random_state=42, n_jobs=-1)),
    ("RSTSF-Unsupervised",     lambda: RSTSFUnsupervised(n_estimators=200, n_intervals=50, random_state=42, n_jobs=-1)),
    ("RSTSF-UnsupervisedRaw",  lambda: RSTSFUnsupervisedRaw(n_intervals=50, random_state=42, n_jobs=-1)),
]


In [ ]:
results = []

for dataset in DATASETS:
    X_train, y_train, X_test, y_test = data_loader.load_fold(dataset, fold=5)
    print(f"\n{dataset}: train={X_train.shape}, test={X_test.shape}")

    for clf_name, clf_fn in CLASSIFIERS:
        clf = clf_fn()
        t0 = time.perf_counter()
        clf.fit(X_train, y_train)
        fit_time = time.perf_counter() - t0

        t0 = time.perf_counter()
        acc = accuracy_score(y_test, clf.predict(X_test))
        pred_time = time.perf_counter() - t0

        results.append({
            "dataset": dataset,
            "classifier": clf_name,
            "accuracy": acc,
            "fit_s": round(fit_time, 1),
            "predict_s": round(pred_time, 2),
        })
        print(f"  {clf_name:22s}  acc={acc:.4f}  fit={fit_time:.1f}s")


ArrowHead: train=(36, 1, 251), test=(175, 1, 251)
  RSTSF                   acc=0.8514  fit=2.8s
  RSTSF-Random            acc=0.8400  fit=4.8s
  RSTSF-Unsupervised      acc=0.8857  fit=30.2s
  RSTSF-UnsupervisedRaw   acc=0.8286  fit=9.2s

Coffee: train=(28, 1, 286), test=(28, 1, 286)
  RSTSF                   acc=1.0000  fit=2.7s
  RSTSF-Random            acc=0.9643  fit=4.7s
  RSTSF-Unsupervised      acc=1.0000  fit=35.1s
  RSTSF-UnsupervisedRaw   acc=1.0000  fit=10.9s

BME: train=(30, 1, 128), test=(150, 1, 128)
  RSTSF                   acc=1.0000  fit=2.2s
  RSTSF-Random            acc=1.0000  fit=4.5s
  RSTSF-Unsupervised      acc=0.8467  fit=11.7s


In [ ]:
df = pd.DataFrame(results)

# Accuracy pivot
acc_table = df.pivot(index="dataset", columns="classifier", values="accuracy")
acc_table = acc_table[["RSTSF", "RSTSF-Random", "RSTSF-Unsupervised", "RSTSF-UnsupervisedRaw"]]
print("=== Accuracy ===")
print(acc_table.to_string(float_format="{:.4f}".format))

print()

# Fit time pivot
time_table = df.pivot(index="dataset", columns="classifier", values="fit_s")
time_table = time_table[["RSTSF", "RSTSF-Random", "RSTSF-Unsupervised", "RSTSF-UnsupervisedRaw"]]
print("=== Fit time (s) ===")
print(time_table.to_string(float_format="{:.1f}".format))


In [ ]:
# Mean accuracy per classifier
print("Mean accuracy:")
print(df.groupby("classifier")["accuracy"].mean().reindex(["RSTSF", "RSTSF-Random", "RSTSF-Unsupervised", "RSTSF-UnsupervisedRaw"]).to_string(float_format="{:.4f}".format))

print()
print("Mean fit time (s):")
print(df.groupby("classifier")["fit_s"].mean().reindex(["RSTSF", "RSTSF-Random", "RSTSF-Unsupervised", "RSTSF-UnsupervisedRaw"]).to_string(float_format="{:.1f}".format))
